In [2]:

"""
04_shap_maps.ipynb — 12-city SHAP spatial maps

For each city:
  - Load geometry from data/<city>_rq1_compare.gpkg
  - Join SHAP values + gap_z from modeling_table
  - Plot 4-panel map: gap_z + top-3 SHAP dims
  - Save → outputs/figures/city_shap_map/<city>_shap_map.png

Also moves existing shap_summary PNGs to outputs/figures/city_shap_summary/
"""

import geopandas as gpd, pandas as pd, numpy as np
import matplotlib, shutil, contextily as ctx
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

ROOT    = Path("..").resolve()   # rq1_explainability/
OUT     = ROOT / "outputs"
FIG_MAP = OUT / "figures" / "city_shap_map"
FIG_SUM = OUT / "figures" / "city_shap_summary"
FIG_MAP.mkdir(parents=True, exist_ok=True)
FIG_SUM.mkdir(parents=True, exist_ok=True)

# ── Load modeling table (for gap_z) ──────────────────────────────────────────
mt = pd.read_csv(OUT / "modeling_table.csv", dtype={"GEOID": str})
mt["GEOID"] = mt["GEOID"].str.zfill(11)

CITIES = sorted(mt["city"].unique().tolist())
EMBED_COLS = [f"A{i:02d}" for i in range(64)]

# ── SHAP map loop ─────────────────────────────────────────────────────────────
for city in CITIES:
    shap_path = OUT / f"{city}_shap_values.csv"
    gpkg_path = ROOT / "data" / f"{city}_rq1_compare.gpkg"

    if not shap_path.exists():
        print(f"{city}: no SHAP file, skip")
        continue
    if not gpkg_path.exists():
        print(f"{city}: no GPKG, skip")
        continue

    print(f"{city.upper()}")

    # Load geometry
    gdf = gpd.read_file(gpkg_path)[["GEOID", "geometry"]].copy()
    gdf["GEOID"] = gdf["GEOID"].astype(str).str.zfill(11)
    gdf = gdf.to_crs(4326)

    # Load SHAP values
    shap_df = pd.read_csv(shap_path, dtype={"GEOID": str})
    shap_df["GEOID"] = shap_df["GEOID"].str.zfill(11)

    # Top 3 SHAP dims
    shap_cols = [c for c in shap_df.columns if c != "GEOID"]
    top3 = shap_df[shap_cols].abs().mean().sort_values(ascending=False).head(3).index.tolist()
    print(f"  top SHAP dims: {top3}")

    # Merge: geometry + SHAP + gap_z
    city_mt = mt[mt["city"] == city][["GEOID", "hi_minus_lst_z"]].copy()
    joined = (gdf
              .merge(shap_df[["GEOID"] + top3], on="GEOID", how="left")
              .merge(city_mt, on="GEOID", how="left"))

    print(f"  joined rows: {len(joined)}, gap_z notna: {joined.hi_minus_lst_z.notna().sum()}")

    # ── 4-panel plot ──────────────────────────────────────────────────────────
    plot_cols   = ["hi_minus_lst_z"] + top3
    plot_titles = ["HI−LST gap (z-score)"] + [f"SHAP: {d}" for d in top3]

    fig, axes = plt.subplots(1, 4, figsize=(28, 7))
    fig.suptitle(f"{city.replace('_', ' ').title()} — SHAP Spatial Maps", fontsize=15, y=1.01)

    for ax, col, title in zip(axes, plot_cols, plot_titles):
        sub = joined.dropna(subset=[col])
        vmax = sub[col].abs().quantile(0.98)
        vmin = -vmax

        # background (all tracts grey)
        joined.plot(ax=ax, color="#e8e8e8", edgecolor="none")
        sub.plot(column=col, ax=ax, cmap="PuOr",
                 vmin=vmin, vmax=vmax, alpha=0.85,
                 legend=True, legend_kwds={"shrink": 0.6})
        try:
            ctx.add_basemap(ax, crs="EPSG:4326",
                            source=ctx.providers.OpenStreetMap.Mapnik,
                            alpha=0.4)
        except Exception as e:
            print(f"      basemap warning: {e}")
        ax.set_title(title, fontsize=11)
        ax.axis("off")

    plt.tight_layout()
    out_path = FIG_MAP / f"{city}_shap_map.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  saved → figures/city_shap_map/{city}_shap_map.png")

print(f"Maps   → {FIG_MAP}")
print(f"Summary plots → {FIG_SUM}")


FileNotFoundError: [Errno 2] No such file or directory: '/Users/chenchenmengmeng/Documents/Development/Projects/Sigspatial/heat-exposure-hotspot-mismatch/outputs/modeling_table.csv'

In [7]:

"""
PCA spatial maps — 12 cities
For each city:
  - Load PC scores from outputs/tables/<city>_pc_scores.csv
  - Join with GPKG geometry
  - Plot PC1-PC4 as 4-panel map with OSM basemap
  - Save → outputs/figures/city_pca_map/<city>_pca_map.png
"""

import geopandas as gpd, pandas as pd, numpy as np
import matplotlib, contextily as ctx
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

ROOT     = Path("..").resolve()
OUT      = ROOT/"outputs"
FIG_PCA  = OUT / "figures" / "city_pca_map"
FIG_PCA.mkdir(parents=True, exist_ok=True)

mt     = pd.read_csv(ROOT /"data"/"processed"/"modeling_table.csv", dtype={"GEOID": str})
mt["GEOID"] = mt["GEOID"].str.zfill(11)
CITIES = sorted(mt["city"].unique().tolist())

for city in CITIES:
    pc_path   = OUT / "tables" / f"{city}_pc_scores.csv"
    gpkg_path = ROOT / "data" / "processed"/ f"{city}_rq1_compare.gpkg"

    if not pc_path.exists():
        print(f"{city}: no PC scores, skip"); continue
    if not gpkg_path.exists():
        print(f"{city}: no GPKG, skip"); continue

    print(f"\n{city.upper()}")

    gdf = gpd.read_file(gpkg_path)[["GEOID", "geometry"]].copy()
    gdf["GEOID"] = gdf["GEOID"].astype(str).str.zfill(11)
    gdf = gdf.to_crs(4326)

    pc_df = pd.read_csv(pc_path, dtype={"GEOID": str})
    pc_df["GEOID"] = pc_df["GEOID"].str.zfill(11)

    city_mt = mt[mt["city"] == city][["GEOID", "hi_minus_lst_z"]].copy()

    joined = (gdf
              .merge(pc_df, on="GEOID", how="left")
              .merge(city_mt, on="GEOID", how="left"))

    pc_cols    = ["PC1", "PC2", "PC3", "PC4"]
    available  = [c for c in pc_cols if c in joined.columns]
    if not available:
        print(f"  no PC cols found, skip"); continue

    fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 7))
    if len(available) == 1:
        axes = [axes]
    fig.suptitle(f"{city.replace('_', ' ').title()} — PC Score Maps", fontsize=15, y=1.01)

    for ax, col in zip(axes, available):
        sub  = joined.dropna(subset=[col])
        vmax = sub[col].abs().quantile(0.98)
        vmin = -vmax

        joined.plot(ax=ax, color="#e8e8e8", edgecolor="none")
        sub.plot(column=col, ax=ax, cmap="PuOr",
                 vmin=vmin, vmax=vmax, alpha=0.85,
                 legend=True, legend_kwds={"shrink": 0.6})
        try:
            ctx.add_basemap(ax, crs="EPSG:4326",
                            source=ctx.providers.OpenStreetMap.Mapnik,
                            alpha=0.4)
        except Exception as e:
            print(f"  basemap warning: {e}")
        ax.set_title(col, fontsize=12)
        ax.axis("off")

    plt.tight_layout()
    out_path = FIG_PCA / f"{city}_pca_map.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  saved → figures/city_pca_map/{city}_pca_map.png")

print("\nAll PCA maps done.")



ATLANTA
  saved → figures/city_pca_map/atlanta_pca_map.png

BOSTON
  saved → figures/city_pca_map/boston_pca_map.png

CHICAGO
  saved → figures/city_pca_map/chicago_pca_map.png

DALLAS
  saved → figures/city_pca_map/dallas_pca_map.png

HOUSTON
  saved → figures/city_pca_map/houston_pca_map.png

LAS_VEGAS
  saved → figures/city_pca_map/las_vegas_pca_map.png

LOS_ANGELES
  saved → figures/city_pca_map/los_angeles_pca_map.png

MIAMI
  saved → figures/city_pca_map/miami_pca_map.png

NEW_YORK
  saved → figures/city_pca_map/new_york_pca_map.png

PHOENIX
  saved → figures/city_pca_map/phoenix_pca_map.png

SAN_FRANCISCO
  saved → figures/city_pca_map/san_francisco_pca_map.png

SEATTLE
  saved → figures/city_pca_map/seattle_pca_map.png

All PCA maps done.
